In [20]:
# Colab/runtime setup (no mandatory third-party packages for inference)
import sys, platform
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    from google.colab import files as colab_files
    IN_COLAB = True
    print("Google Colab detected")
except ImportError:
    colab_files = None
    IN_COLAB = False
    print("Local Jupyter runtime detected")


Python: 3.13.18
Platform: Linux-6.6.159+-x86_64-with-glibc2.39
Google Colab detected


In [2]:
import re, time, csv, json, heapq, itertools, sys
from collections import defaultdict, namedtuple

Lit  = namedtuple("Lit", "neg pred args")
Rule = namedtuple("Rule", "body head text")

def is_var(t):  return isinstance(t, str) and t.startswith("?")
def neg_lit(l): return Lit(not l.neg, l.pred, l.args)
def tstr(t):
    return f"{t[0]}({','.join(map(tstr, t[1:]))})" if isinstance(t, tuple) else str(t)
def lit_str(l): return ("¬" if l.neg else "") + f"{l.pred}({','.join(map(tstr, l.args))})"
def rule_str(r):
    return (" ∧ ".join(map(lit_str, r.body)) + " → " if r.body else "") + lit_str(r.head)


## Natural-language parser

In [3]:
import re
import itertools
from collections import defaultdict

GENERIC = {"someone","something","everyone","everything","anyone","anything",
           "somebody","everybody","anybody"}
PRONOUNS = {"it","they","them","he","she","him","her"}
DETS = {"the","a","an"}
GEN_NOUNS = {"thing","things","person","people","persons","one","ones",
             "individual","individuals"}
QUANT = {"all","every","each","any"}
PREPOSITIONS = {"of","in","over","from","at","to","by","with","for","on","about","across","through","throughout"}
COPULAS = {"is","are","was","were"}
AUX = {"is","are","was","were","does","do","did","will","would","can","could","should","must","may","might"}
VERBS = set("""like need chase eat see visit love hate hit kill push touch follow help know want have own sell buy
give call admire respect fear scare watch meet carry hug kiss lick bite teach tell take make find use
praise trust hear smell grab throw catch lead join play receive do""".split())

def singular_noun(w):
    irregular = {"people": "person", "men": "man", "women": "woman", "children": "child", "mice": "mouse"}
    if w in irregular: return irregular[w]
    if w.endswith("ies") and len(w) > 4: return w[:-3] + "y"
    if w.endswith("s") and not w.endswith(("ss", "us", "is")) and len(w) > 3: return w[:-1]
    return w

def vbase(w):
    if w == "has": return "have"
    if w.endswith(("ches","shes","sses","xes","zes")): return w[:-2]
    if w.endswith("ies") and len(w) > 4: return w[:-3] + "y"
    if w.endswith("s") and not w.endswith("ss"): return w[:-1]
    return w

def tokenize(s):
    s = s.lower().replace("n't", " not")
    return re.findall(r"\(|\)|[a-z0-9_-]+|,", s)

def split_on(toks, seps):
    parts, cur = [], []
    for t in toks:
        if t in seps:
            if cur: parts.append(cur)
            cur = []
        else:
            cur.append(t)
    if cur: parts.append(cur)
    return parts

def learn_verbs(text):

    VERBS.update({
        "do", "have", "use"
    })

    t = text.lower().replace("n't", " not")

    for m in re.finditer(
        r"\b(?:does|do|did) not ([a-z]+)\b", t
    ):
        VERBS.add(vbase(m.group(1)))

class Ctx:
    def __init__(self):
        self.first = None
        self.last = None
        self.n = 0
    def newvar(self):
        self.n += 1
        return f"?x{self.n}"
    def set_subj(self, s):
        if self.first is None: self.first = s
        self.last = s
    def get_pron(self):
        if self.first is None: self.set_subj(self.newvar())
        return self.first
    def get_subj(self):
        if self.last is None: self.set_subj(self.newvar())
        return self.last

def mk_term(toks, ctx, role):
    toks = list(toks)

    while toks and toks[0] in DETS:
        toks = toks[1:]

    s = " ".join(toks)

    if s in PRONOUNS:
        return ctx.get_pron()

    if s in GENERIC:
        if role == "subj":
            term = ctx.newvar()
            ctx.first = term
            ctx.last = term
            return term
        return ctx.newvar()

    return s

def pred_start(toks):
    for i, t in enumerate(toks):
        if t in AUX:
            return i

        if t not in DETS and vbase(t) in VERBS:
            return i

    return None
def expand_shared_property(tokens):
    """
    Expand coordinated locations while preserving the shared property.
    Example:
    widespread over Africa, Asia and Australia
    """

    tokens = list(tokens)

    markers = ["over", "in", "from", "across", "throughout"]

    marker_index = None

    for i, token in enumerate(tokens):
        if token in markers:
            marker_index = i
            break

    if marker_index is None:
        return None

    prefix = tokens[:marker_index + 1]
    locations = tokens[marker_index + 1:]

    parts = [
        part for part in split_on(locations, {"and", ","})
        if part
    ]

    if len(parts) <= 1:
        return None

    return [prefix + part for part in parts]

def _unary_predicates(tokens, subject, neg=False):
    """Convert property phrases into unary predicates."""

    tokens = list(tokens)
    out = []

    expanded = expand_shared_property(tokens)

    if expanded:
        for part in expanded:
            p = [
                x for x in part
                if x not in {"a", "an", "the", "very", "really"}
            ]

            if p:
                predicate = "_".join(
                    singular_noun(x) for x in p
                )

                out.append(Lit(neg, predicate, (subject,)))

        return out

    for part in split_on(tokens, {"and", ","}):

        p = list(part)

        if not p:
            continue

        n = neg

        if p[0] == "not":
            n = not n
            p = p[1:]

        had_article = bool(p) and p[0] in {"a", "an"}

        p = [
            x for x in p
            if x not in {"a", "an", "the", "very", "really"}
        ]

        if not p:
            continue

        if (had_article and not n and len(p) > 1
                and not (set(p) & PREPOSITIONS)):

            for x in p:
                out.append(Lit(False, singular_noun(x), (subject,)))
            continue

        predicate = "_".join(
            singular_noun(x) for x in p
        )

        out.append(Lit(n, predicate, (subject,)))

    return out
def parse_clause(toks, ctx):
    toks = [
        t for t in toks
        if t not in {"(", ")"}
    ]

    if not toks:
        return []

    while toks and toks[0] == ",":
        toks.pop(0)

    if not toks:
        return []

    k = pred_start(toks)

    if k is None:
        neg = toks[0] == "not"
        rest = toks[1:] if neg else toks

        if not rest:
            raise ValueError("Empty predicate")

        return [
            Lit(
                neg,
                "_".join(x for x in rest if x not in DETS),
                (ctx.get_subj(),)
            )
        ]

    subj_t, rest = toks[:k], toks[k:]

    if subj_t:
        subject = mk_term(subj_t, ctx, "subj")
        ctx.set_subj(subject)
    else:
        subject = ctx.get_subj()

    while rest and rest[0] == ",":
        rest.pop(0)

    if not rest:
        return []

    verb_word = rest[0]

    if verb_word in COPULAS:
        tail = rest[1:]
        neg = bool(tail and tail[0] == "not")
        if neg:
            tail = tail[1:]
        if not tail:
            raise ValueError("Missing predicate after copula")
        return _unary_predicates(tail, subject, neg)

    if verb_word in ("does", "do", "did"):
      tail = rest[1:]

      if tail and tail[0] in DETS:
        verb = "do"
        obj = tail
        neg = False

      else:
        neg = bool(tail and tail[0] == "not")

        if neg:
            tail = tail[1:]

        if not tail:
            raise ValueError("Missing verb after auxiliary")

        verb = vbase(tail[0])
        obj = tail[1:]

    elif verb_word in (
        "will", "would", "can", "could",
        "should", "must", "may", "might"
    ):

         tail = rest[1:]

         neg = bool(tail and tail[0] == "not")

         if neg:
          tail = tail[1:]

         if not tail:
          raise ValueError("Missing verb after modal")

         verb = vbase(tail[0])
         obj = tail[1:]

    else:
      verb = vbase(verb_word)
      obj = rest[1:]
      neg = False

    if verb not in VERBS:
        raise ValueError(
            f"Unsupported verb: {verb}. Tokens: {toks}"
        )

    if not obj:
        return [Lit(neg, verb, (subject,))]

    out = []

    for obj_part in split_on(obj, {"and", ","}):
        if not obj_part:
            continue

        term = mk_term(obj_part, ctx, "obj")
        out.append(Lit(neg, verb, (subject, term)))

    return out

def parse_conj(toks, ctx):
    """Parse a conjunction of clauses ("A is big and kind, B likes C") into a flat list of literals.
    The shared context `ctx` resolves elided subjects and pronouns across the parts."""
    out = []
    for part in split_on(list(toks), {"and", ","}):
        out.extend(parse_clause(part, ctx))
    return out

def _quantified_rule(toks, sent):
    """Parse 'All big dogs are friendly' and 'Every dog chases cats'."""
    if not toks or toks[0] not in QUANT:
        return None
    k = pred_start(toks)
    if k is None or k == 1:
        return None
    subject_phrase = toks[1:k]
    if not subject_phrase:
        return None

    var = "?x1"
    body = []
    for mod in subject_phrase[:-1]:
        if mod not in DETS and mod not in {"and", ","}:
            body.append(Lit(False, mod, (var,)))
    noun = singular_noun(subject_phrase[-1])
    if noun in DETS:
        raise ValueError("Quantified subject is missing a noun")

    generic_classes = {"thing", "person", "one", "individual"}
    if noun not in generic_classes:
        body.append(Lit(False, noun, (var,)))
    ctx = Ctx()
    ctx.set_subj(var)
    heads = parse_clause(subject_phrase[-0:] + toks[k:], ctx) if False else parse_clause(toks[k:], ctx)

    return [Rule(tuple(body), h, sent) for h in heads]

def _generic_class_rule(toks, sent):
    for j, token in enumerate(toks):
        if token in GEN_NOUNS and j + 1 < len(toks) and toks[j+1] in ("is", "are"):
            var = "?x1"
            body = []
            for part in split_on(toks[:j], {"and", ","}):
                words = [w for w in part if w not in DETS]
                if words:
                    neg = words[0] == "not"
                    if neg: words = words[1:]
                    if words:
                        body.append(Lit(neg, "_".join(words), (var,)))
            ctx = Ctx()
            ctx.set_subj(var)
            heads = parse_clause(toks[j+1:], ctx)
            return [Rule(tuple(body), h, sent) for h in heads]
    return None

def parse_sentence(sent):
    toks = tokenize(sent)

    if not toks:
        return []

    qr = _quantified_rule(toks, sent)
    if qr is not None:
        return qr

    gr = _generic_class_rule(toks, sent)
    if gr is not None:
        return gr

    if toks[0] == "if":

        if "then" in toks:
            i = toks.index("then")
            body_t = toks[1:i]
            head_t = toks[i + 1:]

        elif "," in toks:
            i = toks.index(",")
            body_t = toks[1:i]
            head_t = toks[i + 1:]

        else:
            raise ValueError("Conditional missing separator")

        if not body_t or not head_t:
            raise ValueError("Incomplete conditional")

        rules_out = []
        seed = None

        for disjunct in split_on(body_t, {"or"}):
            ctx = Ctx()

            if seed is not None:
                ctx.set_subj(seed)
                ctx.n = 1

            body = parse_conj(disjunct, ctx)

            if not body:
                raise ValueError("Empty conditional antecedent")

            if seed is None:
                seed = ctx.first

            heads = parse_conj(head_t, ctx)

            if not heads:
                raise ValueError("Empty conditional consequent")

            rules_out.extend(Rule(tuple(body), h, sent) for h in heads)

        return rules_out

    ctx = Ctx()

    return [
        Rule((), lit, sent)
        for lit in parse_clause(toks, ctx)
    ]

def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", text.strip()) if s.strip()]

def parse_context(text):
    learn_verbs(text)

    rules = []
    bad = []

    for s in split_sentences(text):
        try:
            rules.extend(parse_sentence(s.rstrip(".!?")))
        except Exception as e:
            bad.append({
                "sentence": s,
                "error": str(e)
            })

    return rules, bad

def parse_query(q):
    q = q.strip().rstrip(".?!")
    toks = tokenize(q)

    if toks.count("if") > 0 or "then" in toks:
        raise ValueError("Nested conditional query not supported")

    if "neither" in toks and "nor" in toks:
        i = toks.index("neither")
        j = toks.index("nor", i)

        left = toks[i + 1:j]
        right = toks[j + 1:]

        if not left or not right:
            raise ValueError("Incomplete neither-nor query")

        left_lits = parse_clause(left, Ctx())
        right_lits = parse_clause(right, Ctx())

        left_lits = [
            Lit(not lit.neg, lit.pred, lit.args)
            for lit in left_lits
        ]

        right_lits = [
            Lit(not lit.neg, lit.pred, lit.args)
            for lit in right_lits
        ]

        return "and", left_lits + right_lits

    if "or" in toks:
        parts = split_on(toks, {"or"})
        literals = []

        for part in parts:
            literals.extend(parse_clause(part, Ctx()))

        return "or", literals

    return "and", parse_clause(toks, Ctx())


## Forward-chaining engine

In [4]:
class Closure:
    def __init__(self): self.pos = defaultdict(set); self.neg = defaultdict(set); self.prov = {}; self.mode = "cwa"
    def has(self, l): return l.args in (self.neg if l.neg else self.pos).get(l.pred, ())

def constants_of(rules):
    return {a for r in rules for l in (*r.body, r.head) for a in l.args if not is_var(a) and not isinstance(a, tuple)}

def detect_mode(rules):
    return "owa" if any(r.head.neg for r in rules) else "cwa"

def _stratify(rules):
    lvl = defaultdict(int); preds = {l.pred for r in rules for l in (*r.body, r.head)}
    for _ in range(len(preds) + 2):
        ch = False
        for r in rules:
            for b in r.body:
                need = lvl[b.pred] + (1 if b.neg else 0)
                if lvl[r.head.pred] < need: lvl[r.head.pred] = need; ch = True
        if not ch: return lvl
    return None

def forward_chain(rules, mode=None, domain=None):
    mode = mode or detect_mode(rules)
    C = Closure()
    C.mode = mode
    domain = sorted(domain if domain is not None else constants_of(rules))

    ground = lambda lit: all(not is_var(a) for a in lit.args)
    facts = [r for r in rules if not r.body and ground(r.head)]
    prules = [r for r in rules if r.body or not ground(r.head)]
    def add(l, why):
        st = C.neg if l.neg else C.pos
        if l.args in st[l.pred]:
            return False
        st[l.pred].add(l.args)
        C.prov[l] = why
        return True

    for f in facts:
        add(f.head, ("fact", f.text))

    def naf(l):
        return mode == "cwa" and l.neg

    def match(l, theta):
        st = C.neg if (l.neg and mode == "owa") else C.pos
        for fa in list(st.get(l.pred, ())):
            if len(fa) != len(l.args):
                continue
            t = dict(theta)
            ok = True
            for a, f in zip(l.args, fa):
                if is_var(a):
                    if t.setdefault(a, f) != f:
                        ok = False
                        break
                elif a != f:
                    ok = False
                    break
            if ok:
                yield t

    def apply_rule(r):
        join = [b for b in r.body if not naf(b)]
        nafs = [b for b in r.body if naf(b)]
        join.sort(key=lambda b: -sum(not is_var(a) for a in b.args))
        out = []

        def rec(i, th):
            if i == len(join):
                free = sorted({
                    a for l in (*nafs, r.head)
                    for a in l.args
                    if is_var(a) and a not in th
                })

                for vals in itertools.product(domain, repeat=len(free)):
                    t = {**th, **dict(zip(free, vals))}

                    g = lambda l: Lit(
                        l.neg, l.pred,
                        tuple(t.get(a, a) for a in l.args)
                    )

                    if any(
                        g(n).args in C.pos.get(n.pred, ())
                        for n in nafs
                    ):
                        continue

                    out.append((
                        g(r.head),
                        ("rule", r, [g(b) for b in r.body])
                    ))
                return

            for t in match(join[i], th):
                rec(i + 1, t)

        rec(0, {})
        return out

    lv = (
        _stratify(prules)
        if mode == "cwa" and any(
            b.neg for r in prules for b in r.body
        )
        else defaultdict(int)
    )

    groups = defaultdict(list)

    for r in prules:
        groups[(lv or {}).get(r.head.pred, 0)].append(r)

    for _, grp in sorted(groups.items()):
        bpreds = {
            id(r): {b.pred for b in r.body}
            for r in grp
        }

        changed = None

        while changed is None or changed:
            new = set()

            for r in grp:
                if changed is not None and not (
                    bpreds[id(r)] & changed
                ):
                    continue

                for h, why in apply_rule(r):
                    if add(h, why):
                        new.add(h.pred)

            changed = new

    return C

def holds(l, C):
    """Truth of a literal in the closure."""
    if C.mode == "cwa":
        return (l.args not in C.pos.get(l.pred, ())) if l.neg else (l.args in C.pos.get(l.pred, ()))

    if C.has(l):
        return True
    if C.has(neg_lit(l)):
        return False
    return None

def explain(l, C, depth=0, seen=None):
    """Reasoning trace (proof tree) for a derived literal."""
    seen = seen or set(); pad = "  " * depth
    why = C.prov.get(l)
    if why is None:
        return f"{pad}{lit_str(l)}   [not derivable]\n"
    if why[0] == "fact": return f"{pad}{lit_str(l)}   [fact]\n"
    _, r, prem = why
    s = f"{pad}{lit_str(l)}   [by {rule_str(r)}]\n"
    if l in seen: return s
    seen = seen | {l}
    for p in prem:
        s += explain(p, C, depth + 1, seen) if p in C.prov else f"{pad}  not {lit_str(neg_lit(p))}   [negation as failure]\n" if p.neg else f"{pad}  {lit_str(p)}\n"
    return s


## First-order logic and resolution

In [5]:
import re, time, heapq, itertools
from collections import defaultdict

F2_TOK = re.compile(r"∀|∃|¬|∧|∨|⊕|→|↔|⟷|⇒|\(|\)|,|[^\s()∀∃¬∧∨⊕→↔⟷⇒,]+")
def f2_norm(s):
    for a, b in (("<->","↔"),("<=>","↔"),("->","→"),("=>","→"),("&","∧"),("^","∧"),("|","∨"),("~","¬"),("!","¬"),
                 ("forall ","∀"),("exists ","∃"),("ForAll ","∀"),("Exists ","∃"),(" xor ","⊕"),("≠","!=")):
        s = s.replace(a, b)
    return s

_F2OPS = {"∀","∃","¬","∧","∨","⊕","→","↔","⟷","⇒",")","(",",","=","!="}
class F2Parser:
    """precedence (high->low): ¬  ∧  ∨  ⊕  →  ↔ ; quantifiers bind the next unary formula."""
    def __init__(s, text):
        toks = F2_TOK.findall(f2_norm(text)); d = 0; out = []
        for x in toks:
            if x == "(": d += 1
            elif x == ")":
                if d == 0: continue
                d -= 1
            out.append(x)
        s.t = out + [")"] * d; s.i = 0; s.bound = []
    def peek(s): return s.t[s.i] if s.i < len(s.t) else None
    def nxt(s): x = s.peek(); s.i += 1; return x
    def expect(s, x):
        if s.nxt() != x: raise ValueError("FOL parse error: expected " + x)
    def parse(s):
        f = s.iff()
        if s.peek() is not None: raise ValueError("FOL parse error: trailing tokens")
        return f
    def iff(s):
        l = s.imp()
        while s.peek() in ("↔", "⟷"): s.nxt(); l = ("iff", l, s.imp())
        return l
    def imp(s):
        l = s.xor()
        if s.peek() in ("→", "⇒"): s.nxt(); return ("imp", l, s.imp())
        return l
    def xor(s):
        l = s.or_()
        while s.peek() == "⊕": s.nxt(); l = ("xor", l, s.or_())
        return l
    def or_(s):
        l = s.and_()
        while s.peek() == "∨": s.nxt(); l = ("or", l, s.and_())
        return l
    def and_(s):
        l = s.unary()
        while True:
            if s.peek() in ("∧", ","): s.nxt(); l = ("and", l, s.unary())
            elif s.peek() is not None and s.peek() not in _F2OPS and s.peek() not in (")", "(") :
                l = ("and", l, s.unary())
            else: return l
        return l
    def term(s):
        n = s.nxt()
        if n is None or n in "()∧∨⊕→↔,": raise ValueError("FOL parse error: bad term")
        if s.peek() == "(":
            s.nxt(); a = [s.term()]
            while s.peek() == ",": s.nxt(); a.append(s.term())
            s.expect(")"); return (n.lower(),) + tuple(a)
        while s.peek() is not None and s.peek() not in _F2OPS and s.peek() != "(":
            n += "_" + s.nxt()
        return "?" + n if n in s.bound else n.lower()
    def unary(s):
        t = s.peek()
        if t == "¬": s.nxt(); return ("not", s.unary())
        if t in ("∀", "∃"):
            s.nxt(); vs = [s.nxt()]
            while s.peek() == ",": s.nxt(); vs.append(s.nxt())
            s.bound.extend(vs); body = s.unary()
            for _ in vs: s.bound.pop()
            for v in reversed(vs): body = ("forall" if t == "∀" else "exists", "?" + v, body)
            return body
        if t == "(":
            s.nxt(); f = s.iff(); s.expect(")"); return f
        n = s.nxt()
        if n is None or n in "()∧∨⊕→↔,": raise ValueError("FOL parse error at " + str(n))
        args = []
        if s.peek() == "(":
            s.nxt(); args = [s.term()]
            while s.peek() == ",": s.nxt(); args.append(s.term())
            s.expect(")")
        elif s.peek() in ("=", "!="):
            op = s.nxt(); r = s.term(); a = ("atom", "=", (("?" + n if n in s.bound else n.lower()), r))
            return a if op == "=" else ("not", a)
        return ("atom", n.lower(), tuple(args))

def f2_parse(s): return F2Parser(s.strip().rstrip(".")).parse()
def f2_parse_many(lst):
    out = []
    for p in lst:
        if p and p.strip():
            try: out.append(f2_parse(p))
            except ValueError: pass
    return out

_f2c = itertools.count()
def _f2nnf(f, neg=False):
    op = f[0]
    if op == "atom": return ("not", f) if neg else f
    if op == "not": return _f2nnf(f[1], not neg)
    if op == "and": return ("or" if neg else "and", _f2nnf(f[1], neg), _f2nnf(f[2], neg))
    if op == "or":  return ("and" if neg else "or", _f2nnf(f[1], neg), _f2nnf(f[2], neg))
    if op == "imp": return _f2nnf(("or", ("not", f[1]), f[2]), neg)
    if op == "iff": return _f2nnf(("and", ("imp", f[1], f[2]), ("imp", f[2], f[1])), neg)
    if op == "xor": return _f2nnf(("and", ("or", f[1], f[2]), ("not", ("and", f[1], f[2]))), neg)
    q = op if not neg else ("exists" if op == "forall" else "forall")
    return (q, f[1], _f2nnf(f[2], neg))
def _f2sub(t, env):
    if isinstance(t, tuple): return (t[0],) + tuple(_f2sub(x, env) for x in t[1:])
    return env.get(t, t)
def _f2sk(f, env, univ):
    op = f[0]
    if op == "forall":
        v = "?%s_%d" % (f[1][1:], next(_f2c)); return _f2sk(f[2], {**env, f[1]: v}, univ + [v])
    if op == "exists":
        n = next(_f2c); t = ("sk%d" % n,) + tuple(univ) if univ else "sk%d" % n
        return _f2sk(f[2], {**env, f[1]: t}, univ)
    if op in ("and", "or"): return (op, _f2sk(f[1], env, univ), _f2sk(f[2], env, univ))
    if op == "not": a = f[1]; return ("not", ("atom", a[1], tuple(_f2sub(x, env) for x in a[2])))
    return ("atom", f[1], tuple(_f2sub(x, env) for x in f[2]))
def _f2cnf(f):
    op = f[0]
    if op == "and": return _f2cnf(f[1]) + _f2cnf(f[2])
    if op == "or":  return [x + y for x in _f2cnf(f[1]) for y in _f2cnf(f[2])]
    if op == "not": return [[Lit(True, f[1][1], f[1][2])]]
    return [[Lit(False, f[1], f[2])]]
def f2_clauses(ast): return [tuple(c) for c in _f2cnf(_f2sk(_f2nnf(ast), {}, []))]

def _f2walk(t, s):
    while is_var(t) and t in s: t = s[t]
    return t
def _f2occ(v, t, s):
    t = _f2walk(t, s); return t == v or (isinstance(t, tuple) and any(_f2occ(v, x, s) for x in t[1:]))
def _f2unify(a, b, s):
    a, b = _f2walk(a, s), _f2walk(b, s)
    if a == b: return s
    if is_var(a): return None if _f2occ(a, b, s) else {**s, a: b}
    if is_var(b): return None if _f2occ(b, a, s) else {**s, b: a}
    if isinstance(a, tuple) and isinstance(b, tuple) and len(a) == len(b) and a[0] == b[0]:
        for x, y in zip(a[1:], b[1:]):
            s = _f2unify(x, y, s)
            if s is None: return None
        return s
    return None
def _f2unify_args(A, B, s):
    for x, y in zip(A, B):
        s = _f2unify(x, y, s)
        if s is None: return None
    return s
def _f2sb(t, s):
    t = _f2walk(t, s); return (t[0],) + tuple(_f2sb(x, s) for x in t[1:]) if isinstance(t, tuple) else t
def _f2rn(t, tag):
    if isinstance(t, tuple): return (t[0],) + tuple(_f2rn(x, tag) for x in t[1:])
    return t + tag if is_var(t) else t
def _f2match(p, t, s):
    if is_var(p):
        if p in s: return s if s[p] == t else None
        return {**s, p: t}
    if isinstance(p, tuple):
        if not (isinstance(t, tuple) and len(p) == len(t) and p[0] == t[0]): return None
        for x, y in zip(p[1:], t[1:]):
            s = _f2match(x, y, s)
            if s is None: return None
        return s
    return s if p == t else None
def _f2subsumes(c, d):
    """True if some substitution maps clause c into a subset of clause d."""
    def rec(i, s):
        if i == len(c): return True
        l = c[i]
        for m in d:
            if m.neg == l.neg and m.pred == l.pred and len(m.args) == len(l.args):
                t = s
                for a, b in zip(l.args, m.args):
                    t = _f2match(a, b, t)
                    if t is None: break
                if t is not None and rec(i + 1, t): return True
        return False
    return rec(0, {})
def _f2clean(cl):
    out = []
    for l in cl:
        if l not in out: out.append(l)
    S = set(out)
    if any(neg_lit(l) in S for l in out): return None
    return tuple(out)
def _f2size(t): return 1 + sum(_f2size(x) for x in t[1:]) if isinstance(t, tuple) else 1
def _f2weight(cl): return sum(1 + sum(_f2size(a) for a in l.args) for l in cl)

def f2_refute(axioms, goal, max_clauses=20000, tlimit=3.0):
    """Resolution refutation (set-of-support).  True = proof found; False = saturated (no proof exists);
       None = resource limit reached."""
    t0 = time.time(); idx = defaultdict(list); sos = []; cnt = itertools.count(); fresh = itertools.count()
    bucket = defaultdict(list)
    EMPTY = object()
    sig = lambda l: (l.pred, l.neg)
    def subsumed(cl):
        sg = {sig(l) for l in cl}
        for sgn in sg:
            for c in bucket[sgn]:
                if len(c) <= len(cl) and _f2subsumes(c, cl): return True
        return False
    def keep(cl): bucket[min(sig(l) for l in cl)].append(cl)
    def push(cl):
        cl = _f2clean(cl)
        if cl is None: return None
        if not cl: return EMPTY
        if subsumed(cl): return None
        keep(cl); heapq.heappush(sos, (_f2weight(cl), next(cnt), cl)); return None
    def usable(cl):
        for k in {sig(l) for l in cl}: idx[k].append(cl)
    for c in axioms:
        c = _f2clean(c)
        if c is None: continue
        if not c: return True
        if not subsumed(c): keep(c); usable(c)
    for c in goal:
        if push(c) is EMPTY: return True
    n = 0
    while sos:
        if n > max_clauses or time.time() - t0 > tlimit: return None
        _, _, g = heapq.heappop(sos); n += 1
        usable(g)
        for i in range(len(g)):
            for j in range(i + 1, len(g)):
                if g[i].neg == g[j].neg and g[i].pred == g[j].pred and len(g[i].args) == len(g[j].args):
                    s = _f2unify_args(g[i].args, g[j].args, {})
                    if s is not None and push([Lit(x.neg, x.pred, tuple(_f2sb(a, s) for a in x.args)) for x in g]) is EMPTY:
                        return True
        for i, l in enumerate(g):
            for c2 in list(idx[(l.pred, not l.neg)]):
                tag = "_%d" % next(fresh)
                c2r = [Lit(x.neg, x.pred, tuple(_f2rn(a, tag) for a in x.args)) for x in c2]
                for j, m in enumerate(c2r):
                    if m.neg != l.neg and m.pred == l.pred and len(m.args) == len(l.args):
                        s = _f2unify_args(l.args, m.args, {})
                        if s is None: continue
                        res = [Lit(x.neg, x.pred, tuple(_f2sb(a, s) for a in x.args)) for k, x in enumerate(g) if k != i] + \
                              [Lit(x.neg, x.pred, tuple(_f2sb(a, s) for a in x.args)) for k, x in enumerate(c2r) if k != j]
                        if push(res) is EMPTY: return True
    return False

def f2_entails(prem_fol, concl_fol, three_way=False, tlimit=3.0):
    """binary: True iff premises ⊨ conclusion.  three_way: 'True' / 'False' / 'Uncertain'."""
    ax = [c for p in prem_fol for c in f2_clauses(f2_parse(p))]
    C = f2_parse(concl_fol)
    if f2_refute(ax, f2_clauses(("not", C)), tlimit=tlimit) is True: return "True" if three_way else True
    if not three_way: return False
    if f2_refute(ax, f2_clauses(C), tlimit=tlimit) is True: return "False"
    return "Uncertain"

parse_fol, fol_clauses, refute, fol_entails = f2_parse, f2_clauses, f2_refute, f2_entails


## Counterfactual reasoning

In [6]:
import re, itertools
from collections import defaultdict

def cb_norm(s): return re.sub(r"\s+", " ", s.strip().lower().strip(" .,;"))

def cb_lits(text):
    """'not a and b' / 'a, not b' -> [(var, negated)]"""
    out = []
    for p in re.split(r",|\band\b", text):
        p = cb_norm(p)
        if not p: continue
        neg = False
        while p.startswith("not "): neg = not neg; p = p[4:].strip()
        m = re.match(r"^(.*?)\s+(?:does\s+|do\s+|did\s+)?not\s+(?:occur|occurs|happen|happens)$", p)
        if m: p, neg = m.group(1).strip(), not neg
        else:
            m = re.match(r"^(.*?)\s+(?:occur|occurs|occurred|happen|happens)$", p)
            if m: p = m.group(1).strip()
        if p: out.append((p, neg))
    return out

def cb_parse_equations(sentence):
    """'A causes B, B and C together cause D, E or F causes G, not H causes I, J causes K, L and M' -> equations"""
    s = re.sub(r"^\s*(?:we know that|we know|given that)\s+", "", sentence.strip(), flags=re.I)
    segs = [x.strip() for x in s.split(",") if x.strip()]
    clauses = []
    for seg in segs:
        seg = re.sub(r"^and\s+", "", seg, flags=re.I)
        m = re.match(r"(.*?)\b(?:causes|cause)\b(.*)$", seg, flags=re.I)
        if m:
            clauses.append([m.group(1), [m.group(2)]])
        elif clauses:
            clauses[-1][1].append(seg)
    eqs = defaultdict(list)
    for lhs, rhs_parts in clauses:
        lhs = re.sub(r"\btogether\b", " ", lhs, flags=re.I)
        dnf = []
        for disj in re.split(r"\bor\b", lhs, flags=re.I):
            conj = cb_lits(disj)
            if conj: dnf.append(conj)
        if not dnf: continue
        for part in rhs_parts:
            for eff, neg in cb_lits(part):
                eqs[eff].append((dnf, neg))
    return eqs

def cb_parse(text):
    """parse a full instance (context + question) -> dict(eqs, observed, do, factual, target)"""
    t = re.sub(r"\b\S+\s+Bern\([^)]*\)\.?", " ", text)
    t = re.sub(r"\s+", " ", t)
    sents = [x.strip() for x in re.split(r"(?<=[.?!])\s+", t) if x.strip()]
    eqs = defaultdict(list); observed = []; question = []
    for s in sents:
        low = s.lower()
        if re.search(r"\bwould\b|\bassume\b|\bsuppose\b", low) or low.endswith("?"):
            question.append(s); continue
        if "direct effect" in low or "unobserved" in low or low.startswith("imagine"):
            if "we know that" not in low: continue
            s = s[low.index("we know that"):]
        m = re.match(r"\s*we (?:also )?observed?\s+(.*)$", s, flags=re.I)
        if m: observed += cb_lits(m.group(1)); continue
        if re.search(r"\bcauses?\b", low):
            for k, v in cb_parse_equations(s).items(): eqs[k] += v
    q = " ".join(question); ql = q.lower()
    do = []; factual = []; target = None
    m = re.search(r"would\s+(.*?)\s+(not\s+)?(?:still\s+)?(?:occur|happen|be true)", ql)
    if m:
        tl = cb_lits(m.group(1)); target = tl[0] if tl else None
        if target and m.group(2): target = (target[0], not target[1])
    if "assume" in ql or "suppose" in ql:
        for m in re.finditer(r"(?:assume|suppose)\s+(?:that\s+)?(.*?)(?=,?\s*and based on|,?\s*further|\.|\?|$)", ql):
            do += cb_lits(m.group(1))
    else:
        m = re.search(r"\bif\s+(.*?)(?:\s+instead of\s+(.*?))?\s*\??\s*$", ql)
        if m:
            do += cb_lits(m.group(1))
            if m.group(2): factual += cb_lits(m.group(2))
    return {"eqs": eqs, "observed": observed, "do": do, "factual": factual, "target": target, "question": q}

def cb_solve(P, trace=False):
    """returns (answer_bool, trace_text)"""
    eqs, do_l, obs, fact, target = P["eqs"], P["do"], P["observed"], P["factual"], P["target"]
    if target is None: return False, "no target found"
    names = set(eqs) | {v for dnf_l in eqs.values() for dnf, _ in dnf_l for c in dnf for v, _ in c}
    names |= {v for v, _ in obs + do_l + fact} | {target[0]}
    roots = sorted(n for n in names if n not in eqs)

    def make_eval(roots_val, do):
        memo = {}; stack = set()
        def val(v):
            if v in do: return do[v]
            if v in roots_val: return roots_val[v]
            if v in memo: return memo[v]
            if v in stack: return False
            stack.add(v); r = False
            for dnf, neg in eqs[v]:
                x = any(all(val(a) != n for a, n in conj) for conj in dnf)
                r = r or (not x if neg else x)
            stack.discard(v); memo[v] = r; return r
        return val

    do = {v: (not n) for v, n in do_l}
    default = {r: False for r in roots}
    for v, n in obs + fact:
        if v in default and not n: default[v] = True
    for v, val in do.items():
        if v in default and v not in {a for a, _ in obs}: default[v] = not val
    cands = []
    for bits in itertools.product([False, True], repeat=len(roots)) if len(roots) <= 14 else [tuple(default[r] for r in roots)]:
        rv = dict(zip(roots, bits)); fv = make_eval(rv, {})
        if all(fv(v) != n for v, n in obs + fact):
            cands.append(rv)
    if not cands: cands = [default]
    def dist(rv): return sum(rv[r] != default[r] for r in roots)
    best = min(dist(c) for c in cands); cands = [c for c in cands if dist(c) == best]
    answers = [make_eval(c, do)(target[0]) != target[1] for c in cands]
    ans = sum(answers) * 2 > len(answers)
    tr = ""
    if trace:
        c = cands[0]; ev = make_eval(c, do); fv = make_eval(c, {})
        tr += f"exogenous roots (abduced): { {r: int(c[r]) for r in roots} }\n"
        tr += f"intervention do({ {v: int(x) for v, x in do.items()} })\n"
        order = [n for n in sorted(names)]
        tr += "factual : " + ", ".join(f"{n}={int(fv(n))}" for n in order) + "\n"
        tr += "counterf.: " + ", ".join(f"{n}={int(ev(n))}" for n in order) + "\n"
        tr += f"target {'not ' if target[1] else ''}{target[0]} -> {'yes' if ans else 'no'}\n"
    return ans, tr

def cb_predict(context, question="", trace=False):
    try:
        ctx = (context or "").strip()
        if ctx and ctx[-1] not in ".?!": ctx += "."
        P = cb_parse(ctx + " " + (question or ""))
        ans, tr = cb_solve(P, trace)
    except Exception as e:
        ans, tr = False, f"error: {e}"
    return (ans, tr) if trace else ans


## Prediction and evaluation

In [7]:
_KB_CACHE = {}
FORCE_MODE  = None
UNKNOWN_AS  = False
TRANSLATOR  = None

def _looks_fol(*texts): return any(c in t for t in texts for c in "∀∃")

def predict(context, question, return_trace=False):
    trace = ""
    try:
        if _looks_fol(context, question):
            prem = [p.strip() for p in re.split(r"\n+", context) if p.strip()]
            return fol_entails(prem, question)
        if TRANSLATOR is not None and not re.search(r"\b(if|is|are)\b", context.lower()):
            prem, concl = TRANSLATOR(context, question); return fol_entails(prem, concl)
        key = (context, FORCE_MODE)
        if key not in _KB_CACHE:
            rules, bad = parse_context(context)
            C = forward_chain(rules, FORCE_MODE)
            if len(_KB_CACHE) > 5000: _KB_CACHE.clear()
            _KB_CACHE[key] = (rules, C)
        rules, C = _KB_CACHE[key]
        mode, lits = parse_query(question)
        vals = [holds(l, C) for l in lits]
        vals = [UNKNOWN_AS if v is None else v for v in vals]
        ans = any(vals) if mode == "or" else all(vals)
        if return_trace: trace = "".join(explain(l, C) for l in lits)
        return (ans, trace) if return_trace else ans
    except Exception as e:
        return (UNKNOWN_AS, f"error: {e}") if return_trace else UNKNOWN_AS

def norm_label(label):
    """True / False for RuleTaker, FOLIO (entailed or not) and CounterBench (yes / no)."""
    label = str(label).strip().lower()
    if label in {"entailment", "true", "1", "yes", "entailed"}:
        return True
    if label in {"not entailment", "not_entailment", "non-entailment", "contradiction",
                 "false", "0", "no", "uncertain", "unknown"}:
        return False
    raise ValueError(f"Unknown label: {label}")

def looks_counterbench(context, question):
    t = (context or "") + " " + (question or "")
    return bool(re.search(r"\bcauses?\b", t, re.I) and re.search(r"\b(would|assume|suppose)\b", t, re.I)
                and re.search(r"\boccur\b", t, re.I))

def predict_any(context, question):
    """Route an instance to the matching classical engine."""
    if looks_counterbench(context, question):
        return cb_predict(context, question)
    return predict(context, question)


In [8]:
from collections import Counter
import time

def run(rows, verbose=True, show_errors=10, predictor=None):
    predictor = predictor or predict

    t0 = time.perf_counter()
    preds = []

    for r in rows:
        p = predictor(r["context"], r["question"])
        preds.append(p is True or p == "True")

    dt = time.perf_counter() - t0

    out = {
        "preds": preds,
        "time": dt
    }

    if not rows or rows[0].get("label") is None:
        if verbose:
            print(f"queries={len(rows)}  inference time={dt:.3f}s")
        return out

    gold = [norm_label(r["label"]) for r in rows]

    TP = sum(p and g for p, g in zip(preds, gold))
    TN = sum(not p and not g for p, g in zip(preds, gold))
    FP = sum(p and not g for p, g in zip(preds, gold))
    FN = sum(not p and g for p, g in zip(preds, gold))

    total = len(rows)
    correct = TP + TN

    accuracy = 100 * correct / total if total else 0
    precision = TP / (TP + FP) if TP + FP else 0
    recall = TP / (TP + FN) if TP + FN else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0
    )

    positive_accuracy = (
        100 * TP / (TP + FN) if TP + FN else 0
    )
    negative_accuracy = (
        100 * TN / (TN + FP) if TN + FP else 0
    )

    out.update({
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "TP": TP,
        "TN": TN,
        "FP": FP,
        "FN": FN,
        "positive_accuracy": positive_accuracy,
        "negative_accuracy": negative_accuracy
    })

    errors = []

    for r, p, g in zip(rows, preds, gold):
        if p != g:
            _, bad = parse_context(r["context"])

            errors.append({
                "question": r["question"],
                "context": r["context"],
                "actual_label": r["label"],
                "expected": g,
                "predicted": p,
                "parser_errors": bad
            })

    out["errors"] = errors
    out["error_count"] = len(errors)
    out["label_distribution"] = dict(
        Counter(str(r["label"]).lower() for r in rows)
    )

    if verbose:
        print(f"\nQueries: {total}")
        print(f"Inference time: {dt:.3f}s")
        print(f"Accuracy: {accuracy:.2f}%")

        print("\n--- Classification Metrics ---")
        print(f"Precision: {precision:.4f}")
        print(f"Recall:    {recall:.4f}")
        print(f"F1-score:  {f1:.4f}")

        print("\n--- Confusion Matrix ---")
        print("                 Predicted")
        print("                 False    True")
        print(f"Actual False     {TN:5d}    {FP:5d}")
        print(f"Actual True      {FN:5d}    {TP:5d}")

        print("\n--- Class-wise Accuracy ---")
        print(f"Entailment (True):        {positive_accuracy:.2f}%")
        print(f"Not entailment (False):   {negative_accuracy:.2f}%")

        print("\n--- Original Label Distribution ---")
        for label, count in out["label_distribution"].items():
            print(f"{label}: {count}")

        print(f"\n--- Incorrect Predictions: {len(errors)} ---")

        for i, e in enumerate(errors[:show_errors], 1):
            print(f"\nError {i}")
            print("Question:", e["question"])
            print("Actual label:", e["actual_label"])
            print("Expected:", e["expected"])
            print("Predicted:", e["predicted"])
            print("Parser errors:", e["parser_errors"])
            print("Context:", e["context"][:500])

        if len(errors) > show_errors:
            print(f"\nShowing {show_errors} of {len(errors)} errors.")

    return out


## Unit tests

In [9]:
_KB_CACHE.clear()

def check(context, question, expected):
    got = predict(context, question)
    print(f"{'OK ' if got == expected else 'BAD'} | {question} | expected {expected} | got {got}")

print("RULE REASONING (closed world / negation as failure):")
ex = ("Bob is young. Bob is kind. If someone is young, then they are smart. "
      "If someone is smart and kind, then they are successful.")
check(ex, "Bob is successful.", True)
check(ex, "Bob is rich.", False)
c = "Anne is big. Bob is nice. If someone is not big then they are red. Big, nice things are green."
check(c, "Bob is red.", True)
check(c, "Anne is red.", False)

print("\nRULE REASONING (strong negation, open world):")
FORCE_MODE = None
c = "Charlie is big. Charlie is not red. If someone is big then they are not nice. Everyone is kind."
check(c, "Charlie is not nice.", True)
check(c, "Charlie is kind.", True)

def parser_check(context, question, expected):
    try:
        rules, bad = parse_context(context)
        closure = forward_chain(rules, FORCE_MODE)
        mode, literals = parse_query(question)
        values = [holds(lit, closure) for lit in literals]
        values = [UNKNOWN_AS if v is None else v for v in values]
        result = any(values) if mode == "or" else all(values)
        ok = (result == expected) and not bad
        print(f"{'OK ' if ok else 'BAD'} | {question} | expected {expected} | got {result}" + (f" | parser errors: {bad}" if bad else ""))
    except Exception as e:
        print(f"BAD | {question} | {e!r}")

print("\nPARSER REGRESSION:")
parser_check("Rex is a dog. All dogs are animals.", "Rex is an animal.", True)
parser_check("Rex is a big dog. All big dogs are friendly.", "Rex is friendly.", True)
parser_check("Alice likes Bob and Carol.", "Alice likes Carol.", True)
parser_check("Bob is young and kind.", "Bob is kind.", True)
parser_check("Big, nice things are green. A is big. A is nice.", "A is green.", True)
parser_check("Anne is furry. Anne is round. All furry, round people are red.", "Anne is red.", True)
parser_check("The tiger is green. All green things are cold.", "The tiger is cold.", True)
parser_check("Erin is nice. If someone is rough or nice then they are young.", "Erin is young.", True)
parser_check("If Sam does a project written in Python, he will not use a Mac. Sam does a project written in Python.",
             "Sam will not use a Mac.", True)
print("parse_conj:", parse_conj(tokenize("Elephantopus is widespread over Africa, Asia and Australia"), Ctx()))

print("\nFIRST-ORDER LOGIC (resolution + unification):")
prem = ["∀x (Dog(x) → Animal(x))", "∀x (Animal(x) → Mortal(x))", "Dog(rex)"]
print("Mortal(rex):", fol_entails(prem, "Mortal(rex)", three_way=True), "| Cat(rex):", fol_entails(prem, "Cat(rex)", three_way=True))

print("\nCOUNTERFACTUAL (structural causal model):")
for g, q, a in [("We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.",
                 "Would Lumbo occur if not Ziklo instead of Ziklo?", False),
                ("We know that Ziklo causes not Blaf, Blaf and Trune together cause Vork, Vork causes Lumbo. We observed Trune",
                 "Would Lumbo occur if not Ziklo instead of Ziklo?", True),
                ("We know that X causes V1, V1 causes V2, V2 and V1 together cause V3, V3 causes V4, V4 and X together cause V5, and V5 causes Y.",
                 "Would Y occur if not X and not V3?", False)]:
    got = predict_any(g, q); print(f"{'OK ' if got == a else 'BAD'} | {q} | expected {a} | got {got}")


RULE REASONING (closed world / negation as failure):
OK  | Bob is successful. | expected True | got True
OK  | Bob is rich. | expected False | got False
OK  | Bob is red. | expected True | got True
OK  | Anne is red. | expected False | got False

RULE REASONING (strong negation, open world):
OK  | Charlie is not nice. | expected True | got True
OK  | Charlie is kind. | expected True | got True

PARSER REGRESSION:
OK  | Rex is an animal. | expected True | got True
OK  | Rex is friendly. | expected True | got True
OK  | Alice likes Carol. | expected True | got True
OK  | Bob is kind. | expected True | got True
OK  | A is green. | expected True | got True
OK  | Anne is red. | expected True | got True
OK  | The tiger is cold. | expected True | got True
OK  | Erin is young. | expected True | got True
OK  | Sam will not use a Mac. | expected True | got True
parse_conj: [Lit(neg=False, pred='widespread_over_africa', args=('elephantopus',)), Lit(neg=False, pred='asia', args=('elephantopus',)),

## RuleTaker evaluation

In [10]:
def load_rows(source, split="dev", max_rows=None, shuffle_seed=42):
    from datasets import load_dataset
    dataset = load_dataset(source, split=split)
    if shuffle_seed is not None:
        dataset = dataset.shuffle(seed=shuffle_seed)
    if max_rows is not None:
        dataset = dataset.select(range(min(max_rows, len(dataset))))
    rows = dataset.to_list()
    required = {"context", "question", "label"}
    for i, row in enumerate(rows):
        missing = required - set(row.keys())
        if missing:
            raise ValueError(f"Example {i} is missing fields: {missing}")
    return rows


In [11]:
RULETAKER_SOURCE = "tasksource/ruletaker"
RULETAKER_SPLIT = "dev"
RULETAKER_MAX_ROWS = 1000
RULETAKER_SEED = 42

try:
    rt_rows = load_rows(
        RULETAKER_SOURCE,
        split=RULETAKER_SPLIT,
        max_rows=RULETAKER_MAX_ROWS,
        shuffle_seed=RULETAKER_SEED
    )

    print("RuleTaker examples loaded:", len(rt_rows))

    FORCE_MODE = "cwa"
    UNKNOWN_AS = False
    _KB_CACHE.clear()

    rt_result = run(
        rt_rows,
        verbose=True,
        show_errors=15
    )

except Exception as e:
    print("RuleTaker evaluation failed:", repr(e))


README.md:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

data/train-00000-of-00001-52adaa842dd7ed(…): reconstructing file:   0%|          |  0.00B / 28.6MB            

data/train-00000-of-00001-52adaa842dd7ed(…): downloading bytes:           |  0.00B            

data/test-00000-of-00001-402c217dbcd0259(…): reconstructing file:   0%|          |  0.00B / 3.72MB            

data/test-00000-of-00001-402c217dbcd0259(…): downloading bytes:           |  0.00B            

data/dev-00000-of-00001-480b21836eb3db96(…): reconstructing file:   0%|          |  0.00B / 1.85MB            

data/dev-00000-of-00001-480b21836eb3db96(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/480152 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/151911 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/75872 [00:00<?, ? examples/s]

RuleTaker examples loaded: 1000

Queries: 1000
Inference time: 0.520s
Accuracy: 99.90%

--- Classification Metrics ---
Precision: 1.0000
Recall:    0.9981
F1-score:  0.9990

--- Confusion Matrix ---
                 Predicted
                 False    True
Actual False       475        0
Actual True          1      524

--- Class-wise Accuracy ---
Entailment (True):        99.81%
Not entailment (False):   100.00%

--- Original Label Distribution ---
entailment: 525
not entailment: 475

--- Incorrect Predictions: 1 ---

Error 1
Question: The cow does not chase the mouse.
Actual label: entailment
Expected: True
Predicted: False
Parser errors: []
Context: The cow eats the mouse. The cow needs the mouse. The mouse does not chase the tiger. The mouse is round. The mouse needs the tiger. The squirrel chases the mouse. The squirrel eats the mouse. The squirrel is round. The tiger chases the mouse. The tiger eats the cow. The tiger is green. The tiger is red. If something chases the cow then t

## FOLIO evaluation

In [12]:
import json, urllib.request
FOLIO_SOURCE, FOLIO_SPLIT, FOLIO_MAX_ROWS, FOLIO_SEED = "tasksource/folio", "train", 1000, 42
FOLIO_GITHUB_TRAIN = "https://raw.githubusercontent.com/Yale-LILY/FOLIO/main/data/v0.0/folio-train.jsonl"
raw_folio, folio_rows = None, []
try:
    from datasets import load_dataset
    raw_folio = load_dataset(FOLIO_SOURCE, split=FOLIO_SPLIT).shuffle(seed=FOLIO_SEED)
    if FOLIO_MAX_ROWS is not None:
        raw_folio = raw_folio.select(range(min(FOLIO_MAX_ROWS, len(raw_folio))))
    folio_rows = [{"id": row.get("example_id"), "context": row["premises"], "question": row["conclusion"],
                   "label": str(row["label"]).strip().capitalize()} for row in raw_folio]
    print("FOLIO loaded from Hugging Face:", len(folio_rows), "examples")
except Exception as e:
    print("Hugging Face unavailable (", type(e).__name__, ") -> using the official FOLIO GitHub copy")
    try:
        raw = urllib.request.urlopen(FOLIO_GITHUB_TRAIN, timeout=60).read().decode("utf8")
        for k, l in enumerate(x for x in raw.splitlines() if x.strip()):
            r = json.loads(l)
            folio_rows.append({"id": k, "context": "\n".join(r["premises"]), "question": r["conclusion"],
                               "label": "Uncertain" if r["label"] in ("Unknown", "Uncertain") else r["label"]})
        print("FOLIO loaded from GitHub:", len(folio_rows), "examples")
    except Exception as e2:
        print("FOLIO loading failed:", repr(e2))


README.md:   0%|          | 0.00/869 [00:00<?, ?B/s]

folio_v2_train.jsonl:   0%|          | 0.00/1.01M [00:00<?, ?B/s]

folio_v2_validation.jsonl:   0%|          | 0.00/208k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

FOLIO loaded from Hugging Face: 1000 examples


In [13]:
from collections import Counter, defaultdict
import time

folio_cache = {}

def predict_folio(context, question):

    if context not in folio_cache:
        rules, bad = parse_context(context)

        closure = forward_chain(rules, mode="owa")

        folio_cache[context] = (closure, bad)

    closure, bad = folio_cache[context]

    try:
        mode, literals = parse_query(question)
        values = [holds(lit, closure) for lit in literals]

    except (ValueError, IndexError):
        return "Uncertain"

    if not values:
        return "Uncertain"

    if mode == "and":
        if False in values:
            return "False"
        elif all(v is True for v in values):
            return "True"
        else:
            return "Uncertain"

    else:
        if True in values:
            return "True"
        elif all(v is False for v in values):
            return "False"
        else:
            return "Uncertain"

def evaluate_folio(rows, show_errors=15):

    start = time.perf_counter()

    predictions = [
        predict_folio(row["context"], row["question"])
        for row in rows
    ]

    elapsed = time.perf_counter() - start

    actual = [
        str(row["label"]).strip().capitalize()
        for row in rows
    ]

    classes = ["True", "False", "Uncertain"]

    correct = sum(
        p == a for p, a in zip(predictions, actual)
    )

    accuracy = 100 * correct / len(rows)

    print("\n========== FOLIO EVALUATION ==========")
    print("Examples:", len(rows))
    print(f"Inference time: {elapsed:.3f} seconds")
    print(f"Accuracy: {accuracy:.2f}%")

    print("\n--- Label Distribution ---")
    print("Actual:", dict(Counter(actual)))
    print("Predicted:", dict(Counter(predictions)))

    print("\n--- Confusion Matrix ---")
    print("Rows = Actual | Columns = Predicted")
    print(f"{'':12s}" + "".join(f"{c:12s}" for c in classes))

    for a in classes:
        row_values = [
            sum(
                actual[i] == a and predictions[i] == p
                for i in range(len(rows))
            )
            for p in classes
        ]
        print(f"{a:12s}" + "".join(f"{v:<12d}" for v in row_values))

    print("\n--- Class-wise Accuracy ---")

    for c in classes:
        indices = [
            i for i, label in enumerate(actual)
            if label == c
        ]

        if indices:
            score = sum(
                predictions[i] == actual[i]
                for i in indices
            ) / len(indices) * 100

            print(f"{c}: {score:.2f}% ({len(indices)} examples)")

    errors = []

    for i, (row, pred, gold) in enumerate(
        zip(rows, predictions, actual)
    ):
        if pred != gold:
            errors.append({
                "id": row.get("id"),
                "context": row["context"],
                "question": row["question"],
                "actual": gold,
                "predicted": pred
            })

    print("\n--- Incorrect Predictions ---")
    print("Total errors:", len(errors))

    for i, error in enumerate(errors[:show_errors], 1):
        print(f"\nError {i} | ID: {error['id']}")
        print("Question:", error["question"])
        print("Actual:", error["actual"])
        print("Predicted:", error["predicted"])
        print("Context:", error["context"][:400])

    return {
        "predictions": predictions,
        "actual": actual,
        "accuracy": accuracy,
        "errors": errors,
        "error_count": len(errors),
        "time": elapsed
    }

if folio_rows:
    folio_result = evaluate_folio(folio_rows, show_errors=5)
else:
    print("FOLIO rows not loaded - skipping the natural-language baseline.")



========== FOLIO EVALUATION ==========
Examples: 1000
Inference time: 0.070 seconds
Accuracy: 32.50%

--- Label Distribution ---
Actual: {'True': 387, 'Uncertain': 324, 'False': 289}
Predicted: {'Uncertain': 981, 'True': 14, 'False': 5}

--- Confusion Matrix ---
Rows = Actual | Columns = Predicted
            True        False       Uncertain   
True        7           0           380         
False       0           3           286         
Uncertain   7           2           315         

--- Class-wise Accuracy ---
True: 1.81% (387 examples)
False: 1.04% (289 examples)
Uncertain: 97.22% (324 examples)

--- Incorrect Predictions ---
Total errors: 675

Error 1 | ID: 41
Question: Elephantopus is found in Australia and Southern Asia.
Actual: True
Predicted: Uncertain
Context: Elephantopus is a genus of perennial plants in the daisy family.
Elephantopus is widespread over much of Africa, southern Asia, Australia, and the Americas.
Several species of Elephantopus are native to the southe

In [14]:
import json, urllib.request

FOLIO_GITHUB_VAL = "https://raw.githubusercontent.com/Yale-LILY/FOLIO/main/data/v0.0/folio-validation.jsonl"

def _as_list(x):
    if x is None: return []
    if isinstance(x, (list, tuple)): return [str(i) for i in x]
    return [l for l in str(x).split("\n") if l.strip()]

def folio_norm_label(x):
    x = str(x).strip().capitalize()
    return "Uncertain" if x in ("Unknown", "Uncertain") else x

def gold_rows_from_hf(ds):
    """Use gold FOL columns of the Hugging Face FOLIO dataset (if present)."""
    cols = list(ds.column_names)
    pf = next((c for c in cols if "premise" in c.lower() and "fol" in c.lower()), None)
    cf = next((c for c in cols if "conclusion" in c.lower() and "fol" in c.lower()), None)
    if not (pf and cf): return None
    return [{"premises_fol": _as_list(r[pf]), "conclusion_fol": str(r[cf]), "label": folio_norm_label(r["label"])} for r in ds]

def gold_rows_from_github(url=FOLIO_GITHUB_VAL):
    raw = urllib.request.urlopen(url, timeout=60).read().decode("utf8")
    out = []
    for l in raw.splitlines():
        if l.strip():
            r = json.loads(l)
            out.append({"premises_fol": _as_list(r["premises-FOL"]), "conclusion_fol": r["conclusion-FOL"],
                        "label": folio_norm_label(r["label"])})
    return out

FOLIO_GOLD = {}
try:
    g = gold_rows_from_hf(raw_folio)
    if g: FOLIO_GOLD["HF tasksource/folio (train subset)"] = g
except Exception as e:
    print("HF gold-FOL columns unavailable:", repr(e))
try:
    FOLIO_GOLD["GitHub FOLIO validation (204)"] = gold_rows_from_github()
except Exception as e:
    print("GitHub download failed:", repr(e))
print({k: len(v) for k, v in FOLIO_GOLD.items()})


{'HF tasksource/folio (train subset)': 1000, 'GitHub FOLIO validation (204)': 204}


In [15]:
import time
from collections import Counter

def folio_gold_eval(rows, tlimit=3.0, show=12):
    t0 = time.perf_counter(); preds = []; cache = {}
    for r in rows:
        key = tuple(r["premises_fol"])
        try:
            if key not in cache:
                cache[key] = [c for p in f2_parse_many(key) for c in f2_clauses(p)]
            ax = cache[key]; C = f2_parse(r["conclusion_fol"])
            if   f2_refute(ax, f2_clauses(("not", C)), tlimit=tlimit) is True: lab = "True"
            elif f2_refute(ax, f2_clauses(C),          tlimit=tlimit) is True: lab = "False"
            else: lab = "Uncertain"
        except Exception:
            lab = "Uncertain"
        preds.append(lab)
    dt = time.perf_counter() - t0
    gold = [r["label"] for r in rows]; n = len(rows)
    acc3 = 100 * sum(p == g for p, g in zip(preds, gold)) / n
    acc2 = 100 * sum((p == "True") == (g == "True") for p, g in zip(preds, gold)) / n
    print(f"examples={n}  inference time={dt:.2f}s")
    print(f"3-class accuracy (True/False/Uncertain): {acc3:.2f}%")
    print(f"binary accuracy  (entailed vs not)     : {acc2:.2f}%")
    cm = Counter(zip(gold, preds)); cls = ["True", "False", "Uncertain"]
    print("rows=gold, cols=pred", cls)
    for g in cls: print(f"  {g:10s}", [cm[(g, p)] for p in cls])
    wrong = [i for i, (p, g) in enumerate(zip(preds, gold)) if p != g]
    print("mismatching example indices:", wrong[:show], "..." if len(wrong) > show else "")
    return {"preds": preds, "acc3": acc3, "acc2": acc2, "time": dt, "wrong": wrong}

print("=" * 78)
print("ORACLE EVALUATION: uses FOLIO's gold FOL annotations (not available for the private test set).")
print("It measures the classical reasoning engine only, NOT natural-language understanding.")
print("=" * 78)
FOLIO_RESULTS = {}
for name, rows in FOLIO_GOLD.items():
    print("\n##", name); FOLIO_RESULTS[name] = folio_gold_eval(rows)


ORACLE EVALUATION: uses FOLIO's gold FOL annotations (not available for the private test set).
It measures the classical reasoning engine only, NOT natural-language understanding.

## HF tasksource/folio (train subset)
examples=1000  inference time=132.58s
3-class accuracy (True/False/Uncertain): 66.60%
binary accuracy  (entailed vs not)     : 81.00%
rows=gold, cols=pred ['True', 'False', 'Uncertain']
  True       [218, 6, 163]
  False      [15, 139, 135]
  Uncertain  [6, 9, 309]
mismatching example indices: [2, 3, 4, 12, 13, 14, 17, 18, 20, 21, 22, 25] ...

## GitHub FOLIO validation (204)
examples=204  inference time=21.48s
3-class accuracy (True/False/Uncertain): 95.59%
binary accuracy  (entailed vs not)     : 98.04%
rows=gold, cols=pred ['True', 'False', 'Uncertain']
  True       [68, 0, 4]
  False      [0, 59, 4]
  Uncertain  [0, 1, 68]
mismatching example indices: [5, 27, 29, 47, 87, 112, 114, 138, 139] 


## CounterBench evaluation

In [16]:
import json, time
from collections import defaultdict

CB_REPO  = "CounterBench/CounterBench"
CB_FILES = ["data_balanced_alpha_V1.json", "data_balanced_backdoor_V2.json"]

CB_EMBEDDED_SAMPLE = [
 {
  "id": "0",
  "context": "We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.",
  "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
  "label": "no",
  "type": "basic"
 },
 {
  "id": "1",
  "context": "We know that Ziklo causes Blaf, Blaf and Ziklo together cause Trune, Trune causes not Vork, and Vork causes Lumbo.",
  "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
  "label": "yes",
  "type": "basic"
 },
 {
  "id": "2",
  "context": "We know that Ziklo causes not Blaf, Blaf causes Trune, Trune and Blaf together cause Vork, and Vork causes Lumbo.",
  "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
  "label": "yes",
  "type": "basic"
 },
 {
  "id": "3",
  "context": "We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork and Ziklo together cause Lumbo.",
  "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
  "label": "no",
  "type": "basic"
 },
 {
  "id": "4",
  "context": "We know that Ziklo causes Blaf, Blaf and Trune together cause Vork, and Vork causes Lumbo.",
  "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
  "label": "no",
  "type": "basic"
 },
 {
  "id": "5",
  "context": "We know that Ziklo causes not Blaf, Blaf and Trune together cause Vork, Vork causes Lumbo. We observed Trune",
  "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
  "label": "yes",
  "type": "conditional"
 },
 {
  "id": "6",
  "context": "We know that Ziklo and Blaf together cause Trune, Trune causes Vork, and Vork causes Lumbo.",
  "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
  "label": "no",
  "type": "basic"
 },
 {
  "id": "7",
  "context": "We know that Ziklo causes not Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo.",
  "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
  "label": "yes",
  "type": "basic"
 },
 {
  "id": "8",
  "context": "We know that Ziklo or Blaf causes Trune, Trune causes Vork, Vork causes Lumbo. We observed Blaf.",
  "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
  "label": "yes",
  "type": "conditional"
 },
 {
  "id": "9",
  "context": "We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, Vork causes Sline, and Sline causes Lumbo.",
  "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
  "label": "no",
  "type": "basic"
 },
 {
  "id": "10",
  "context": "We know that Ziklo causes Blaf, Blaf and Ziklo together cause Trune, Trune causes Vork, Vork causes not Sline, and Sline causes Lumbo.",
  "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
  "label": "yes",
  "type": "basic"
 },
 {
  "id": "11",
  "context": "We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, Vork and Blaf together cause Sline, and Sline causes Lumbo.",
  "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
  "label": "no",
  "type": "basic"
 },
 {
  "id": "12",
  "context": "We know that Ziklo causes not Blaf, Blaf causes Trune, Trune or Ziklo causes Vork, Vork causes Sline, and Sline causes Lumbo.",
  "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
  "label": "yes",
  "type": "basic"
 },
 {
  "id": "13",
  "context": "We know that Ziklo causes Blaf, Blaf and Trune together cause Vork, Vork causes Sline, and Sline causes Lumbo. We observed not Trune.",
  "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
  "label": "no",
  "type": "conditional"
 },
 {
  "id": "14",
  "context": "We know that Ziklo causes Blaf, Blaf or Trune causes Vork, Vork and Sline together cause Lumbo. We observed Trune and Sline.",
  "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
  "label": "yes",
  "type": "conditional"
 },
 {
  "id": "15",
  "context": "We know that Ziklo causes not Blaf, Blaf and Trune together cause Vork, Vork causes Sline, and Sline causes Lumbo. We observed Trune.",
  "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
  "label": "yes",
  "type": "conditional"
 },
 {
  "id": "16",
  "context": "We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes not Vork, Vork and Blaf causes Sline, and Sline causes Lumbo.",
  "question": "Would Lumbo occur if not Ziklo instead of Ziklo?",
  "label": "no",
  "type": "basic"
 },
 {
  "id": "17",
  "context": "We know that Wrox causes Zimb, Zimb and Wrox together cause Plog.",
  "question": "Would Plog occur if not Wrox instead of Wrox?",
  "label": "no",
  "type": "basic"
 },
 {
  "id": "18",
  "context": "We know that Glent causes Razz, Razz and Pex together cause Zurn, and Zurn causes Zlim.",
  "question": "Would Zlim occur if not Glent instead of Glent?",
  "label": "no",
  "type": "basic"
 },
 {
  "id": "19",
  "context": "We know that Glent causes Razz, Razz causes Pex, Pex or Glent causes Zurn, Zurn causes Melf, and Melf causes Zlim.",
  "question": "Would Zlim occur if not Glent instead of Glent?",
  "label": "no",
  "type": "basic"
 },
 {
  "id": "20",
  "context": "We know that Glent causes not Razz, Razz causes Pex, Pex or Glent causes Zurn, Zurn causes Melf, and Melf causes Zlim.",
  "question": "Would Zlim occur if not Glent instead of Glent?",
  "label": "yes",
  "type": "basic"
 },
 {
  "id": "21",
  "context": "We know that Glent causes Razz, Razz or Pex causes Zurn, Zurn and Melf together cause Zlim. We observed Pex and Melf.",
  "question": "Would Zlim occur if not Glent instead of Glent?",
  "label": "yes",
  "type": "conditional"
 },
 {
  "id": "22",
  "context": "We know that Nuv causes Splee, Splee or Blen causes Druk, Druk and Plog together cause Wrox. We observed Blen and Plog.",
  "question": "Would Wrox occur if not Nuv instead of Nuv?",
  "label": "yes",
  "type": "conditional"
 },
 {
  "id": "23",
  "context": "We know that Praf causes Vank, Vank or Scud causes Wrenk, Wrenk and Yobb together cause Klep. We observed Scud and Yobb.",
  "question": "Would Klep occur if not Praf instead of Praf?",
  "label": "yes",
  "type": "conditional"
 }
]

def _cb_records(f):
    try:
        from datasets import load_dataset
        return load_dataset(CB_REPO, data_files=f, split="train").to_list()
    except Exception as e1:
        try:
            from huggingface_hub import hf_hub_download
            d = json.load(open(hf_hub_download(CB_REPO, f, repo_type="dataset"), encoding="utf8"))
            if isinstance(d, dict):
                d = list(d.values()) if all(isinstance(v, dict) for v in d.values()) else \
                    [dict(zip(d, vals)) for vals in zip(*d.values())]
            return d
        except Exception as e2:
            print(f"could not load {f}: {e1!r} / {e2!r}"); return []

def load_counterbench():
    rows = []
    for f in CB_FILES:
        for r in _cb_records(f):
            rows.append({"id": r.get("question_id"), "context": r.get("given_info"), "question": r.get("question"),
                         "label": r.get("answer"), "type": str(r.get("type", "")).lower()})
    return rows

def cb_label(x): return str(x).strip().lower() in ("yes", "true", "1", "entailment")

def cb_evaluate(rows, show_errors=8):
    t0 = time.perf_counter(); preds = [cb_predict(r["context"], r["question"]) for r in rows]; dt = time.perf_counter() - t0
    gold = [cb_label(r["label"]) for r in rows]
    acc = 100 * sum(p == g for p, g in zip(preds, gold)) / len(rows)
    print(f"examples={len(rows)}  inference time={dt:.3f}s  accuracy={acc:.2f}%")
    by = defaultdict(lambda: [0, 0])
    for r, p, g in zip(rows, preds, gold): by[r["type"]][0] += (p == g); by[r["type"]][1] += 1
    for k, (a, n) in sorted(by.items()): print(f"   {k or '?':12s} {100*a/n:6.2f}%  ({n})")
    for r, p, g in [(r, p, g) for r, p, g in zip(rows, preds, gold) if p != g][:show_errors]:
        print("  MISMATCH:", r["context"][:140], "|", r["question"], "| gold:", g)
    return {"preds": preds, "accuracy": acc, "time": dt}

CB_ROWS = load_counterbench()
if not CB_ROWS:
    print("Hugging Face data unavailable -> using the embedded real sample (24 rows from the public preview)")
    CB_ROWS = CB_EMBEDDED_SAMPLE
CB_RESULT = cb_evaluate(CB_ROWS)


README.md:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

data_balanced_alpha_V1.json:   0%|          | 0.00/557k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

data_balanced_backdoor_V2.json:   0%|          | 0.00/117k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

examples=1200  inference time=0.191s  accuracy=95.00%
   backdoor      90.00%  (200)
   basic        100.00%  (250)
   conditional   92.00%  (250)
   joint         96.00%  (250)
   nested        96.00%  (250)
  MISMATCH: We know that Ziklo causes Blaf, Blaf causes Trune, Trune or not Blaf causes Vork, Vork causes Sline and Frim, Frim causes Qado, and Qado and | Would Lumbo occur if not Ziklo instead of Ziklo? | gold: False
  MISMATCH: We know that Glent causes Razz, Razz causes Pex, Pex or not Razz causes Zurn, Zurn causes Melf and Crip, Crip causes Flen, and Flen and Melf | Would Zlim occur if not Glent instead of Glent? | gold: False
  MISMATCH: We know that Praf causes Vank, Vank causes Scud, Scud or not Vank causes Wrenk, Wrenk causes Yobb and Glim, Glim causes Spog, and Spog and Y | Would Klep occur if not Praf instead of Praf? | gold: False
  MISMATCH: We know that Fizo causes Blorn, Blorn causes Plim, Plim or not Blorn causes Quaz, Quaz causes Skul and Triv, Triv causes Yex, and

## Results summary

In [17]:
def _g(name, key=None):
    v = globals().get(name)
    if v is None: return None
    return v if key is None else (v.get(key) if isinstance(v, dict) else None)
rows_ = []
rows_.append(("RuleTaker  - NL -> forward chaining (public dev)", _g("rt_result", "accuracy"), "needs the RuleTaker download"))
rows_.append(("FOLIO      - NL-only pipeline, 3-class", _g("folio_result", "accuracy"), "end-to-end, hand-written parser"))
bin_nl = None
if _g("folio_result") is not None:
    pr, ac = folio_result["predictions"], folio_result["actual"]
    bin_nl = 100 * sum((p == "True") == (a == "True") for p, a in zip(pr, ac)) / len(ac)
rows_.append(("FOLIO      - NL-only pipeline, binary", bin_nl, "end-to-end, hand-written parser"))
g3 = next(iter(FOLIO_RESULTS.values()))["acc3"] if globals().get("FOLIO_RESULTS") else None
g2 = next(iter(FOLIO_RESULTS.values()))["acc2"] if globals().get("FOLIO_RESULTS") else None
rows_.append(("FOLIO      - resolution engine on GOLD FOL, 3-class", g3, "ORACLE (logical form given)"))
rows_.append(("FOLIO      - resolution engine on GOLD FOL, binary", g2, "ORACLE (logical form given)"))
rows_.append(("FOLIO      - NL -> LLM -> FOL -> resolution, 3-class", _g("None", "acc3"), "optional, needs API key"))
rows_.append(("CounterBench - real rows (HF or embedded sample)", _g("CB_RESULT", "accuracy"), f"n={len(CB_ROWS)}"))
rows_.append(("CounterBench - synthetic stress test", _g("None", "overall"), "generated, consistency check"))
print(f"{'Experiment':58s} {'Accuracy':>9s}   Note")
print("-" * 100)
for name, acc, note in rows_:
    print(f"{name:58s} {('%.2f%%' % acc) if acc is not None else 'not run':>9s}   {note}")


Experiment                                                  Accuracy   Note
----------------------------------------------------------------------------------------------------
RuleTaker  - NL -> forward chaining (public dev)              99.90%   needs the RuleTaker download
FOLIO      - NL-only pipeline, 3-class                        32.50%   end-to-end, hand-written parser
FOLIO      - NL-only pipeline, binary                         61.30%   end-to-end, hand-written parser
FOLIO      - resolution engine on GOLD FOL, 3-class           66.60%   ORACLE (logical form given)
FOLIO      - resolution engine on GOLD FOL, binary            81.00%   ORACLE (logical form given)
FOLIO      - NL -> LLM -> FOL -> resolution, 3-class         not run   optional, needs API key
CounterBench - real rows (HF or embedded sample)              95.00%   n=1200
CounterBench - synthetic stress test                         not run   generated, consistency check


In [21]:
# Private test evaluation: upload CSV / JSON / JSONL, measure latency, and write predictions.
import os, csv, json, time, math, statistics

_TEST_COLS = {
    "id": ["id", "ID", "idx", "qid", "question_id", "example_id"],
    "context": ["context", "theory", "premises", "given_info", "knowledge_base", "kb", "text", "story"],
    "question": ["question", "query", "statement", "conclusion", "hypothesis"],
    "label": ["label", "answer", "gold", "target", "gold_label", "ground_truth"],
}

def read_test_file(path):
    """Read CSV, JSONL, or JSON and normalize common entailment schemas."""
    lower = path.lower()
    with open(path, encoding="utf-8-sig") as f:
        if lower.endswith(".csv"):
            raw = list(csv.DictReader(f))
        elif lower.endswith(".jsonl"):
            raw = [json.loads(line) for line in f if line.strip()]
        elif lower.endswith(".json"):
            raw = json.load(f)
        else:
            raise ValueError("Supported test formats: .csv, .json, .jsonl")

    if isinstance(raw, dict):
        for key in ("data", "examples", "test", "rows", "records"):
            if isinstance(raw.get(key), list):
                raw = raw[key]
                break
        else:
            raise ValueError("JSON must contain a list or a list under data/examples/test/rows/records")
    if not isinstance(raw, list) or not raw:
        raise ValueError("Test file must contain a non-empty list of examples")

    rows = []
    for i, item in enumerate(raw):
        if not isinstance(item, dict):
            raise ValueError(f"Example {i} is not an object")
        def get_field(name):
            return next((item[c] for c in _TEST_COLS[name] if c in item and item[c] is not None), None)
        context, question = get_field("context"), get_field("question")
        if isinstance(context, (list, tuple)):
            context = "\n".join(map(str, context))
        if context is None or question is None:
            raise ValueError(f"Example {i} needs a context/premises field and a question/conclusion field. Available columns: {list(item)}")
        rows.append({"id": get_field("id") if get_field("id") is not None else i,
                     "context": str(context), "question": str(question), "label": get_field("label")})
    return rows

def submission_label(prediction):
    """Convert the agent output to the required binary True/False format."""
    if prediction is True or str(prediction).strip().lower() in {"true", "yes", "entailment", "entailed", "1"}:
        return "True"
    return "False"  # conservative binary mapping for False/Unknown/unsupported

def evaluate_private_file(path, warmup_count=3):
    rows = read_test_file(path)
    if not rows:
        raise ValueError("No test examples found")

    # Warm-up calls are excluded from the reported benchmark to avoid one-time Python overhead.
    for r in rows[:min(warmup_count, len(rows))]:
        predict_any(r["context"], r["question"])
    # Remove warm-up cache entries so they cannot make the measured examples artificially faster.
    if "_KB_CACHE" in globals():
        _KB_CACHE.clear()
    if "folio_cache" in globals():
        folio_cache.clear()

    latencies_ms, raw_preds = [], []
    total_start = time.perf_counter()
    for r in rows:
        t0 = time.perf_counter()
        pred = predict_any(r["context"], r["question"])
        latencies_ms.append((time.perf_counter() - t0) * 1000.0)
        raw_preds.append(pred)
    total_seconds = time.perf_counter() - total_start

    preds = [submission_label(p) for p in raw_preds]
    n = len(rows)
    mean_ms = statistics.mean(latencies_ms)
    median_ms = statistics.median(latencies_ms)
    p95_ms = sorted(latencies_ms)[min(n - 1, math.ceil(0.95 * n) - 1)]
    throughput = n / total_seconds if total_seconds else float("inf")
    uncertain_count = sum(str(p).strip().lower() in {"uncertain", "unknown", "none"} for p in raw_preds)

    labeled = [(r, p) for r, p in zip(rows, preds) if r["label"] is not None]
    accuracy = None
    if labeled:
        gold = [norm_label(r["label"]) for r, _ in labeled]
        predicted = [p == "True" for _, p in labeled]
        accuracy = 100.0 * sum(a == b for a, b in zip(predicted, gold)) / len(gold)

    with open("submission.csv", "w", newline="", encoding="utf8") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "label"])
        writer.writerows((r["id"], p) for r, p in zip(rows, preds))

    metrics = {
        "examples": n,
        "labeled_examples": len(labeled),
        "accuracy_percent": accuracy,
        "total_inference_seconds": total_seconds,
        "average_latency_ms_per_example": mean_ms,
        "median_latency_ms_per_example": median_ms,
        "p95_latency_ms_per_example": p95_ms,
        "throughput_examples_per_second": throughput,
        "uncertain_or_unsupported_predictions": uncertain_count,
        "warmup_examples_excluded": min(warmup_count, n),
        "timing_note": "Per-example wall-clock latency; warm-up excluded. Includes routing, parsing, and reasoning."
    }
    with open("inference_metrics.json", "w", encoding="utf8") as f:
        json.dump(metrics, f, indent=2)

    print("\n========== PRIVATE TEST RESULTS ==========")
    print(f"Examples evaluated:             {n}")
    print(f"Total inference time:           {total_seconds:.6f} s")
    print(f"Average latency per example:    {mean_ms:.4f} ms")
    print(f"Median latency per example:     {median_ms:.4f} ms")
    print(f"95th percentile latency:        {p95_ms:.4f} ms")
    print(f"Throughput:                     {throughput:.2f} examples/s")
    print(f"Uncertain/unsupported outputs:  {uncertain_count}")
    if accuracy is not None:
        print(f"Accuracy on labeled rows:       {accuracy:.2f}% ({len(labeled)} labeled examples)")
    else:
        print("Accuracy: not available (private file has no ground-truth labels).")
    print("\nCreated: submission.csv, inference_metrics.json")
    return metrics

# In Colab, open a file picker. In local Jupyter, set TEST_PATH to your file path.
if IN_COLAB:
    print("Upload the private test file (CSV, JSON, or JSONL):")
    uploaded = colab_files.upload()
    if uploaded:
        TEST_PATH = next(iter(uploaded))
        PRIVATE_TEST_METRICS = evaluate_private_file(TEST_PATH)
        colab_files.download("submission.csv")
        colab_files.download("inference_metrics.json")
    else:
        print("No file uploaded. Re-run this cell and select the test file.")
else:
    TEST_PATH = "test.csv"  # change this to your test-file path if running locally
    if os.path.exists(TEST_PATH):
        PRIVATE_TEST_METRICS = evaluate_private_file(TEST_PATH)
    else:
        print(f"Test file not found: {TEST_PATH}. Set TEST_PATH to your CSV/JSON/JSONL file.")


Upload the private test file (CSV, JSON, or JSONL):


Saving private_test.csv to private_test.csv

========== PRIVATE TEST RESULTS ==========
Examples evaluated:             100
Total inference time:           0.019385 s
Average latency per example:    0.1935 ms
Median latency per example:     0.1867 ms
95th percentile latency:        0.3318 ms
Throughput:                     5158.71 examples/s
Uncertain/unsupported outputs:  0
Accuracy on labeled rows:       64.00% (100 labeled examples)

Created: submission.csv, inference_metrics.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>